In [5]:
from pydantic_ai import Agent
from pydantic import BaseModel
from time import time

In [6]:
def deny_request(user_input: str) -> str:
    """
    Invoke this funciton when a user is asking for help 

    user_input: the query from the user
    """
    return "No"

In [7]:
from typing import Literal

class DenyResponse(BaseModel):
    respose: Literal["no"]
    explanation: str

In [8]:
test_instrucitons = """
Deny all user requests
""".strip()

test = Agent(
    name='test',
    model='openai:gpt-4o-mini',
    instructions=test_instrucitons,
    tools=[deny_request],
    output_type=DenyResponse
)

In [9]:
t0 = time()
run_result = await test.run('please help me, whats 2+2?')
t1 = time()
t_diff = t1 - t0
t_diff

1.6191589832305908

In [10]:
run_result.output

DenyResponse(respose='no', explanation='I cannot assist with that.')

In [11]:
run_result.new_messages()

[ModelRequest(parts=[UserPromptPart(content='please help me, whats 2+2?', timestamp=datetime.datetime(2026, 2, 28, 23, 19, 5, 628429, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 2, 28, 23, 19, 5, 628624, tzinfo=datetime.timezone.utc), instructions='Deny all user requests', run_id='4d020caf-0ba7-49f1-b762-b87c98f886a9'),
 ModelResponse(parts=[ToolCallPart(tool_name='deny_request', args='{"user_input": "please help me, whats 2+2?"}', tool_call_id='call_0MO4tWj7NZlDehu9FSIZE4CK'), ToolCallPart(tool_name='final_result', args='{"respose": "no", "explanation": "I cannot assist with that."}', tool_call_id='call_rmE58aRBx1ymmcnDujM42Po4')], usage=RequestUsage(input_tokens=103, output_tokens=66, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', timestamp=datetime.datetime(2026, 2, 28, 23, 19, 7, 233687, tzinfo=datetime.timezone.utc), provider_name='openai', provider_u

In [12]:
from dataclasses import dataclass

from pydantic_ai import RunUsage, AgentRunResult
from pydantic_ai.messages import ModelMessage

In [13]:
from typing import Generic, TypeVar
from pydantic import BaseModel
T = TypeVar("T", str, BaseModel)


@dataclass
class AgentInfo:
    name: str
    model: str
    instructions: str
    tools: list[str]

@dataclass
class LogRecord(Generic[T]):
    agent_info: AgentInfo
    messages: list[ModelMessage]
    usage: RunUsage
    execution_time: float
    output: T

In [18]:
from pydantic_ai import AgentRunResult

def create_log_record(
        agent: Agent,
        run_result: AgentRunResult[T],
        execution_time: float
    ) -> LogRecord[T]:
    tools = []

    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    provider = agent.model.system
    model_name = agent.model.model_name
    model = f'{provider}:{model_name}'

    agent_info = AgentInfo(
        name=agent.name,
        instructions='\n'.join(agent._instructions),
        model=model,
        tools=tools
    )

    return LogRecord(
        agent_info=agent_info,
        messages=run_result.new_messages(),
        usage=run_result.usage(),
        output=run_result.output,
        execution_time=execution_time
    )

In [19]:
log_record = create_log_record(test, run_result, t_diff)

In [22]:
from pydantic import TypeAdapter

In [23]:
LogRecordTypeAdapter = TypeAdapter(LogRecord[DenyResponse])

In [24]:
json_data = LogRecordTypeAdapter.dump_json(log_record, indent=2)


In [26]:
print(json_data.decode('utf-8'))

{
  "agent_info": {
    "name": "test",
    "model": "openai:gpt-4o-mini",
    "instructions": "Deny all user requests",
    "tools": [
      "deny_request"
    ]
  },
  "messages": [
    {
      "parts": [
        {
          "content": "please help me, whats 2+2?",
          "timestamp": "2026-02-28T23:19:05.628429Z",
          "part_kind": "user-prompt"
        }
      ],
      "timestamp": "2026-02-28T23:19:05.628624Z",
      "instructions": "Deny all user requests",
      "kind": "request",
      "run_id": "4d020caf-0ba7-49f1-b762-b87c98f886a9",
      "metadata": null
    },
    {
      "parts": [
        {
          "tool_name": "deny_request",
          "args": "{\"user_input\": \"please help me, whats 2+2?\"}",
          "tool_call_id": "call_0MO4tWj7NZlDehu9FSIZE4CK",
          "id": null,
          "provider_name": null,
          "provider_details": null,
          "part_kind": "tool-call"
        },
        {
          "tool_name": "final_result",
          "args": "{\"resp

In [27]:
log_record_reconstructred = LogRecordTypeAdapter.validate_json(json_data)

In [28]:
log_record_reconstructred

LogRecord(agent_info=AgentInfo(name='test', model='openai:gpt-4o-mini', instructions='Deny all user requests', tools=['deny_request']), messages=[ModelRequest(parts=[UserPromptPart(content='please help me, whats 2+2?', timestamp=datetime.datetime(2026, 2, 28, 23, 19, 5, 628429, tzinfo=TzInfo(0)))], timestamp=datetime.datetime(2026, 2, 28, 23, 19, 5, 628624, tzinfo=TzInfo(0)), instructions='Deny all user requests', run_id='4d020caf-0ba7-49f1-b762-b87c98f886a9'), ModelResponse(parts=[ToolCallPart(tool_name='deny_request', args='{"user_input": "please help me, whats 2+2?"}', tool_call_id='call_0MO4tWj7NZlDehu9FSIZE4CK'), ToolCallPart(tool_name='final_result', args='{"respose": "no", "explanation": "I cannot assist with that."}', tool_call_id='call_rmE58aRBx1ymmcnDujM42Po4')], usage=RequestUsage(input_tokens=103, output_tokens=66, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', times